In [1]:
import pandas as pd

In [2]:
def parse_fasta(filepath, cls: int):
    # 데이터 불러오기
    df = pd.read_csv(filepath, header=None)
    
    # 짝수 인덱스 행(id), 홀수 인덱스 행(sequence)로 분리
    ID = df.iloc[::2].reset_index(drop=True)
    sequence = df.iloc[1::2].reset_index(drop=True)
    
    # 컬럼 이름 변경 및 시퀀스 열 추가
    ID = ID.rename(columns={0: "ID"})
    ID["sequence"] = sequence[0]
    
    # class 열 추가
    ID["class"] = cls
    
    return ID

In [3]:
data_path ='/Users/khj/Desktop/Project_ongoing/DRBP/data/raw_data'

In [4]:
RBP = parse_fasta(f"{data_path}/351_RNAbinding_Protein.fasta.txt", 1)
NRBP = parse_fasta(f"{data_path}/17870_RNANotbinding_Protein.fasta.txt", 0)
DBP = parse_fasta(f"{data_path}/1447_DNAbinding_Protein.fasta.txt", 1)
NDBP = parse_fasta(f"{data_path}/16774_DNANotbinding_Protein.fasta.txt", 0)

In [8]:
print(len(RBP))
print(len(DBP))
print(len(NRBP))
print(len(NDBP))
print(len(RBP)+len(NRBP))
print(len(DBP)+len(NDBP))

351
1447
17870
16774
18221
18221


In [9]:
save_path ='/Users/khj/Desktop/Project_ongoing/DRBP/data/raw_data'

DBP.to_csv(f"{save_path}/DBP.csv",header=True,index=False)
RBP.to_csv(f"{save_path}/RBP.csv",header=True,index=False)
NDBP.to_csv(f"{save_path}/NDBP.csv",header=True,index=False)
NRBP.to_csv(f"{save_path}/NRBP.csv",header=True,index=False)

# 여기부터는 DRBP multi classification을 할 때 사용하는 부분

In [ ]:
def process_data(DBP, RBP, NDBP, NRBP):
    
    # 1. 공통 ID 찾기
    BP_common_ids = set(DBP['ID']) & set(RBP['ID'])
    
    # 2. 공통된 ID를 DRBP로 명명 (DBP와 RBP 중 공통 부분을 합쳐서 만들 수도 있음)
    DRBP = pd.concat([
        DBP[DBP['ID'].isin(BP_common_ids)],
        RBP[RBP['ID'].isin(BP_common_ids)]
    ], ignore_index=True)
    DRBP = DRBP.drop_duplicates(subset=['ID'])

    # 3. DBP와 RBP에서 공통 ID 제거
    DBP = DBP[~DBP['ID'].isin(BP_common_ids)].reset_index(drop=True)
    RBP = RBP[~RBP['ID'].isin(BP_common_ids)].reset_index(drop=True)
    DRBP['class'] = 3
    DBP['class'] = 2
    RBP['class'] = 1
    
    # 4. NRBP에서 공통 ID 제거 및 class 업데이트
    NDRBP = pd.concat([NDBP,NRBP])
    NDRBP = NDRBP.drop_duplicates(subset=['ID'])
    NDRBP['class'] = 0
    NDRBP.drop(NDRBP[NDRBP['ID'].isin(BP_common_ids)].index, inplace=True)
    NDRBP.drop(NDRBP[NDRBP['ID'].isin(DBP['ID'])].index, inplace=True)
    NDRBP.drop(NDRBP[NDRBP['ID'].isin(RBP['ID'])].index, inplace=True)

    NDRBP.reset_index(drop=True, inplace=True)

    return DRBP, DBP, RBP, NDRBP

In [ ]:
data_path ='/Users/khj/Desktop/Project_ongoing/DRBP/data/raw_data'

In [34]:
RNABP = pd.read_csv(f'{data_path}/RBP.csv')
DNABP = pd.read_csv(f'{data_path}/DBP.csv')
NRNABP = pd.read_csv(f'{data_path}/NRBP.csv')
NDNABP = pd.read_csv(f'{data_path}/NDBP.csv')

In [35]:
DRBProt, DBProt, RBProt, NDRBProt = process_data(DNABP, RNABP, NDNABP, NRNABP)

In [36]:
DRBProt

,ID,sequence,class
0,>ENSP00000234170,MAAVKEPLEFHAKRPWRPEEAVEDPDEEDEDNTSEAENGFSLEEVL...,3
1,>ENSP00000400142,MASTDYSTYSQAAAQQGYSAYTAQPTQGYAQTTQAYGQQSYGTYGQ...,3
2,>ENSP00000221419,MSRRLLPRAEKRRRRLEQRQQPDEQRRRSGAMVKMAAAGGGGGGGR...,3
3,>ENSP00000282516,MNGDMPHVPITTLAGIASLTDLLNQLPLPSPLPATTTKSLLFNARI...,3
4,>ENSP00000366396,MGVPAFFRWLSRKYPSIIVNCVEEKPKECNGVKIPVDASKPNPNDV...,3
...,...,...,...
85,>ENSP00000318177,MAELVQGQSAPVGMKAEGFVDALHRVRQIAAKIDSIPHLNNSTPLV...,3
86,>ENSP00000433071,MVKLFIGNLPREATEQEIRSLFEQYGKVLECDIIKNYGFVHIEDKT...,3
87,>ENSP00000456845,MVRERKCILCHIVYSSKKEMDEHMRSMLHHRELENLKGRDISHECR...,3
88,>ENSP00000263063,MAVPGVGLLTRLNLCARRRTRVQRPIVRLLSCPGTVAKDLRRDEQP...,3


In [37]:
DBProt

,ID,sequence,class
0,>ENSP00000363822,MEVQLGLGRVYPRPPSKTYRGAFQNLFQSVREVIQNPGPRHPEAAS...,2
1,>ENSP00000266744,MESSAKMESGGAGQQPQPQPQQPFLPPAACFFATAAAAAAAAAAAA...,2
2,>ENSP00000332293,MDGGTLPRSAPPAPPVPVGCAARRRPASPELLRCSRRRRPATAETG...,2
3,>ENSP00000264110,MKFKLHVNSARQYKDLWNMSDDKPFLCTAPGCGQRFTNEDHLAVHK...,2
4,>ENSP00000286800,MSLSENSVFAYESSVHSTNVLLSLNDQRKKDVLCDVTIFVEGQRFR...,2
...,...,...,...
1352,>ENSP00000263956,MSGFSPELIDYLEGKISFEEFERRREERKTREKKSLQEKGKLSAEE...,2
1353,>ENSP00000404843,MELWGRMLWALLSGPGRRGSTRGWAFSSWQPQPPLAGLSSAIELVS...,2
1354,>ENSP00000229330,MAAPSLLNWRRVSSFTGPVPRARHGHRAVAIRELMIIFGGGNEGIA...,2
1355,>ENSP00000262498,MFKNTFQSGFLSILYSIGSKPLQIWDKKVRNGHIKRITDNDIQSLV...,2


In [38]:
RBProt

,ID,sequence,class
0,>ENSP00000261772,MDSTLTASEIRQRFIDFFKRNEHTYVHSSATIPLDDPTLLFANAGM...,1
1,>ENSP00000477848,MAGPQPLALQLEQLLNPRPSEADPEADPEEATAARVIDRFDEGEDG...,1
2,>ENSP00000313603,MPKAPKQQPPEPEWIGDGESTSPSDKVVKKGKKDKKIKKTFFEELA...,1
3,>ENSP00000274849,MEAEESEKAATEQEPLEGTEQTLDAEEEQEESEEAACGSKKRVVPG...,1
4,>ENSP00000357459,MNPRQGYSLSGYYTHPFQGYEHRQLRYQQPGPGSSPSSFLLKQIEF...,1
...,...,...,...
256,>ENSP00000368017,MATADTPAPASSGLSPKEEGELEDGEISDDDNNSQIRSRSSSSSSG...,1
257,>ENSP00000310042,MATYTCITCRVAFRDADMQRAHYKTDWHRYNLRRKVASMAPVTAEG...,1
258,>ENSP00000467423,MAETLSGLGDSGAAGAAALSSASSETGTRRLSDLRVIDLRAELRKR...,1
259,>ENSP00000376989,MPLRDKYCQTDHHHHGCCEPVYILEPGDPPLLQQPLQTSKSGIQQI...,1


In [39]:
NDRBProt

,ID,sequence,class
0,>ENSP00000489814,MASQNTEQEYEAKLAPSVGGEPTSGGPSGSSPDPNPDSSEVLDRHE...,0
1,>ENSP00000490928,MASQNTEQEYEAKLAPSVGGEPTSGGPSGSSPDPNPDSSEVLDRHE...,0
2,>ENSP00000341108,MGSKEDAGKGCPAAGGVSSFTIQSILGGGPSEAPREPVGWPARKRS...,0
3,>ENSP00000320089,MLLSPVTSTPFSVKDILRLERERSCPAASPHPRVRKSPENFQYLRM...,0
4,>ENSP00000343464,MSDKLKERKVSRLSPNGTCALVVEASDSPTRHLGGPMAGKCPHGTL...,0
...,...,...,...
15267,>ENSP00000332723,MNGFASLLRRNQFILLVLFLLQIQSLGLDIDSRPTAEVCATHTISP...,0
15268,>ENSP00000474570,MDTQGFSCLLLLISEIDLSVKRRI,0
15269,>ENSP00000474861,MATRGFSCLLLVISEIDLSVKRWV,0
15270,>ENSP00000475261,MALKEGLRAWKRIFWRQILLTLGLLGLFLYGLPKFRHLEALIPMGV...,0
